# RAG Answer Relevance Evaluation

This notebook evaluates generated RAG answers with an LLM-as-judge protocol. It uses the existing ground-truth question file, runs the RAG pipeline for each configured answer model, asks a judge model whether each answer is relevant to the question, and summarizes how many answers are relevant per model.

The judge receives the expected ground-truth chunk as reference evidence. By default it evaluates 25 different chunks with one question per chunk to control API cost. Set `RAG_EVAL_ONE_QUESTION_PER_CHUNK=0` to evaluate raw question rows instead, or `RAG_EVAL_MAX_QUESTIONS=0` to remove the row/chunk limit.


In [9]:
from __future__ import annotations

from pathlib import Path
import importlib
import json
import os
import sys
from time import sleep
from typing import Any

import chromadb
from dotenv import load_dotenv
from openai import OpenAI


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


def csv_env(name: str, default: str) -> list[str]:
    return [item.strip() for item in os.getenv(name, default).split(",") if item.strip()]


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env", override=True)
if not (os.getenv("OPENAI_API_KEY") or "").strip():
    raise ValueError("OPENAI_API_KEY is missing. Add it to /workspace/.env or your environment.")

CHROMA_PATH = PROJECT_ROOT / "backend" / "data" / "chroma_db"
GROUND_TRUTH_PATH = PROJECT_ROOT / "backend" / "data" / "eval" / "ground_truth_chunk_questions.jsonl"
OUTPUT_PATH = PROJECT_ROOT / "backend" / "data" / "eval" / "rag_answer_relevance_results.jsonl"
COLLECTION_NAME = os.getenv("RAG_EVAL_COLLECTION", "estate_documents")

# Comma-separated model names make comparisons easy, for example:
# RAG_EVAL_ANSWER_MODELS=gpt-4o-mini,gpt-4o
ANSWER_MODELS = csv_env("RAG_EVAL_ANSWER_MODELS", os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini,gpt-4.1"))  # noqa: E501
JUDGE_MODEL = os.getenv("RAG_EVAL_JUDGE_MODEL", "gpt-4o-mini")
TOP_K = int(os.getenv("RAG_EVAL_TOP_K", "5"))
MAX_QUESTIONS = int(os.getenv("RAG_EVAL_MAX_QUESTIONS", "25"))
ONE_QUESTION_PER_CHUNK = os.getenv("RAG_EVAL_ONE_QUESTION_PER_CHUNK", "1") != "0"
SLEEP_SECONDS = float(os.getenv("RAG_EVAL_SLEEP_SECONDS", "0"))
SAVE_RESULTS = os.getenv("RAG_EVAL_SAVE_RESULTS", "1") != "0"

openai_client = OpenAI()
chroma_client = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

print(f"Project root: {PROJECT_ROOT}")
print(f"Ground truth file: {GROUND_TRUTH_PATH}")
print(f"Chroma collection: {COLLECTION_NAME} ({collection.count()} chunks)")
print(f"Answer models: {ANSWER_MODELS}")
print(f"Judge model: {JUDGE_MODEL}")
print(f"Max questions/chunks: {'all' if MAX_QUESTIONS == 0 else MAX_QUESTIONS}")
print(f"One question per chunk: {ONE_QUESTION_PER_CHUNK}")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Project root: /workspace
Ground truth file: /workspace/backend/data/eval/ground_truth_chunk_questions.jsonl
Chroma collection: estate_documents (93 chunks)
Answer models: ['gpt-4o-mini', 'gpt-4.1']
Judge model: gpt-4o-mini
Max questions/chunks: 25
One question per chunk: True


In [10]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def save_jsonl(rows: list[dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def selected_ground_truth(
    rows: list[dict[str, Any]],
    max_questions: int,
    *,
    one_question_per_chunk: bool = True,
) -> list[dict[str, Any]]:
    selected: list[dict[str, Any]] = []
    seen_chunk_ids: set[str] = set()

    for row in rows:
        chunk_id = str(row.get("expected_chunk_id") or "")
        if one_question_per_chunk:
            if not chunk_id or chunk_id in seen_chunk_ids:
                continue
            seen_chunk_ids.add(chunk_id)

        selected.append(row)
        if max_questions and len(selected) >= max_questions:
            break

    return selected


def fetch_reference_chunks(rows: list[dict[str, Any]], batch_size: int = 100) -> dict[str, dict[str, Any]]:
    chunk_ids = sorted({row["expected_chunk_id"] for row in rows if row.get("expected_chunk_id")})
    references: dict[str, dict[str, Any]] = {}

    for start in range(0, len(chunk_ids), batch_size):
        batch_ids = chunk_ids[start : start + batch_size]
        batch = collection.get(ids=batch_ids, include=["documents", "metadatas"])
        ids = batch.get("ids", [])
        docs = batch.get("documents", [])
        metas = batch.get("metadatas", [])

        for idx, chunk_id in enumerate(ids):
            references[chunk_id] = {
                "chunk_id": chunk_id,
                "text": docs[idx] or "",
                "metadata": metas[idx] or {},
            }

    missing = [chunk_id for chunk_id in chunk_ids if chunk_id not in references]
    if missing:
        print(f"Warning: {len(missing)} expected chunks were not found in Chroma.")
    return references


def compact_sources(sources: list[dict[str, Any]]) -> list[dict[str, Any]]:
    compact = []
    for source in sources:
        compact.append(
            {
                "rank": source.get("rank"),
                "document_id": source.get("document_id"),
                "page_number": source.get("page_number"),
                "chunk_ids": source.get("chunk_ids") or [source.get("chunk_id")],
                "citation_label": source.get("citation_label"),
                "rerank_score": source.get("rerank_score"),
            }
        )
    return compact


ground_truth_rows = load_jsonl(GROUND_TRUTH_PATH)
eval_rows = selected_ground_truth(
    ground_truth_rows,
    MAX_QUESTIONS,
    one_question_per_chunk=ONE_QUESTION_PER_CHUNK,
)
reference_by_chunk = fetch_reference_chunks(eval_rows)

if not eval_rows:
    raise ValueError("No ground-truth rows selected. Run notebooks/generate_ground_truth.ipynb first.")

unique_eval_chunks = {row.get("expected_chunk_id") for row in eval_rows}
print(f"Loaded {len(ground_truth_rows)} ground-truth questions; evaluating {len(eval_rows)} questions.")
print(f"Selected {len(unique_eval_chunks)} unique expected chunks.")
print(f"Loaded {len(reference_by_chunk)} reference chunks.")


Loaded 465 ground-truth questions; evaluating 25 questions.
Selected 25 unique expected chunks.
Loaded 25 reference chunks.


In [11]:
import backend.rag.RAG as RAG

RAG = importlib.reload(RAG)
rag = RAG.rag


def answer_questions(rows: list[dict[str, Any]], models: list[str]) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    total = len(rows) * len(models)
    current = 0

    for model in models:
        print("=" * 80)
        print(f"Answer model: {model}")

        for row in rows:
            current += 1
            print(f"[{current}/{total}] {row['question_id']}")
            base_record = {
                "question_id": row.get("question_id"),
                "question": row.get("question"),
                "expected_chunk_id": row.get("expected_chunk_id"),
                "expected_document_id": row.get("expected_document_id"),
                "document_type": row.get("document_type"),
                "page_number": row.get("page_number"),
                "answer_model": model,
            }

            try:
                result = rag(query=row["question"], model=model, top_k=TOP_K)
                records.append(
                    {
                        **base_record,
                        "answer": result.get("answer", ""),
                        "model_used": result.get("model_used"),
                        "response_time_seconds": result.get("response_time_seconds"),
                        "prompt_tokens": result.get("prompt_tokens"),
                        "completion_tokens": result.get("completion_tokens"),
                        "total_tokens": result.get("total_tokens"),
                        "low_confidence": result.get("low_confidence"),
                        "confidence_warning": result.get("confidence_warning"),
                        "graph_error": result.get("graph_error"),
                        "sources": compact_sources(result.get("sources", [])),
                        "rag_error": None,
                    }
                )
            except Exception as exc:
                records.append({**base_record, "answer": "", "sources": [], "rag_error": str(exc)})

            if SLEEP_SECONDS:
                sleep(SLEEP_SECONDS)

    return records


answer_records = answer_questions(eval_rows, ANSWER_MODELS)
print(f"Generated {len(answer_records)} answer records.")


Answer model: gpt-4o-mini
[1/50] certificate_016-p001-c0000-q01
[2/50] certificate_016-p001-c0001-q01
[3/50] certificate_016-p001-c0002-q01
[4/50] certificate_016-p002-c0003-q01
[5/50] donation_money_006-p001-c0000-q01
[6/50] donation_money_006-p001-c0001-q01
[7/50] donation_money_006-p002-c0002-q01
[8/50] donation_money_006-p002-c0003-q01
[9/50] donation_money_006-p002-c0004-q01
[10/50] donation_money_006-p003-c0005-q01
[11/50] donation_money_007-p001-c0000-q01
[12/50] donation_money_007-p001-c0001-q01
[13/50] donation_money_007-p002-c0002-q01
[14/50] donation_money_007-p002-c0003-q01
[15/50] donation_money_007-p003-c0004-q01
[16/50] donation_money_008-p001-c0000-q01
[17/50] donation_money_008-p001-c0001-q01
[18/50] donation_money_008-p002-c0002-q01
[19/50] donation_money_008-p002-c0003-q01
[20/50] donation_money_008-p003-c0004-q01
[21/50] donation_money_009-p001-c0000-q01
[22/50] donation_money_009-p001-c0001-q01
[23/50] donation_money_009-p002-c0002-q01
[24/50] donation_money_009-p0

In [12]:
JUDGE_SYSTEM_PROMPT = """
You are a strict but fair evaluator for a RAG system over notarial estate documents.
Judge whether the ANSWER is relevant to the QUESTION using the REFERENCE EVIDENCE.

Return only JSON with these keys:
- relevant: boolean
- score: integer from 0 to 3
- rationale: short string
- missing_or_wrong_facts: array of short strings

Rubric:
3 = directly answers the question and is supported by the reference evidence.
2 = mostly answers the question, with only minor incompleteness or harmless extra context.
1 = related to the question but does not answer it, is too vague, or misses important evidence.
0 = irrelevant, contradictory, fabricated, empty, or answers a different question.

Mark relevant=true only for scores 2 or 3. Do not penalize citation formatting.
If the answer says the information is missing even though the reference evidence contains it, score 0 or 1.
""".strip()


def parse_judge_json(content: str) -> dict[str, Any]:
    try:
        data = json.loads(content)
    except json.JSONDecodeError:
        start = content.find("{")
        end = content.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise
        data = json.loads(content[start : end + 1])

    score = int(data.get("score", 0))
    score = max(0, min(3, score))
    issues = data.get("missing_or_wrong_facts") or []
    if isinstance(issues, str):
        issues = [issues]
    return {
        "relevant": bool(data.get("relevant", score >= 2)) and score >= 2,
        "score": score,
        "rationale": str(data.get("rationale", "")).strip(),
        "missing_or_wrong_facts": issues,
    }


def create_judge_completion(messages: list[dict[str, str]]):
    try:
        return openai_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=messages,
            response_format={"type": "json_object"},
        )
    except Exception as exc:
        if "response_format" not in str(exc).lower() and "json" not in str(exc).lower():
            raise
        print("Judge model rejected JSON mode; retrying with prompt-only JSON instructions.")
        return openai_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=messages,
        )


def judge_answer(record: dict[str, Any]) -> dict[str, Any]:
    if record.get("rag_error"):
        return {
            "relevant": False,
            "score": 0,
            "rationale": f"RAG call failed: {record['rag_error']}",
            "missing_or_wrong_facts": ["RAG call failed"],
        }

    reference = reference_by_chunk.get(record.get("expected_chunk_id"), {})
    reference_text = (reference.get("text") or "").strip()
    user_prompt = f"""
QUESTION:
{record.get('question', '')}

ANSWER:
{record.get('answer', '')}

EXPECTED DOCUMENT ID:
{record.get('expected_document_id', '')}

REFERENCE EVIDENCE FROM EXPECTED CHUNK:
{reference_text}
""".strip()

    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    response = create_judge_completion(messages)
    content = response.choices[0].message.content or "{}"
    parsed = parse_judge_json(content)
    usage = response.usage
    parsed.update(
        {
            "judge_model": JUDGE_MODEL,
            "judge_prompt_tokens": int(usage.prompt_tokens if usage else 0),
            "judge_completion_tokens": int(usage.completion_tokens if usage else 0),
            "judge_total_tokens": int(usage.total_tokens if usage else 0),
        }
    )
    return parsed


judged_records: list[dict[str, Any]] = []
for idx, record in enumerate(answer_records, start=1):
    print(f"Judging [{idx}/{len(answer_records)}] {record['answer_model']} {record['question_id']}")
    try:
        judgement = judge_answer(record)
    except Exception as exc:
        judgement = {
            "relevant": False,
            "score": 0,
            "rationale": f"Judge call failed: {exc}",
            "missing_or_wrong_facts": ["Judge call failed"],
            "judge_model": JUDGE_MODEL,
            "judge_prompt_tokens": 0,
            "judge_completion_tokens": 0,
            "judge_total_tokens": 0,
        }
    judged_records.append({**record, **judgement})

    if SLEEP_SECONDS:
        sleep(SLEEP_SECONDS)

if SAVE_RESULTS:
    save_jsonl(judged_records, OUTPUT_PATH)
    print(f"Saved judged records -> {OUTPUT_PATH}")

print(f"Judged {len(judged_records)} answers.")


Judging [1/50] gpt-4o-mini certificate_016-p001-c0000-q01
Judging [2/50] gpt-4o-mini certificate_016-p001-c0001-q01
Judging [3/50] gpt-4o-mini certificate_016-p001-c0002-q01
Judging [4/50] gpt-4o-mini certificate_016-p002-c0003-q01
Judging [5/50] gpt-4o-mini donation_money_006-p001-c0000-q01
Judging [6/50] gpt-4o-mini donation_money_006-p001-c0001-q01
Judging [7/50] gpt-4o-mini donation_money_006-p002-c0002-q01
Judging [8/50] gpt-4o-mini donation_money_006-p002-c0003-q01
Judging [9/50] gpt-4o-mini donation_money_006-p002-c0004-q01
Judging [10/50] gpt-4o-mini donation_money_006-p003-c0005-q01
Judging [11/50] gpt-4o-mini donation_money_007-p001-c0000-q01
Judging [12/50] gpt-4o-mini donation_money_007-p001-c0001-q01
Judging [13/50] gpt-4o-mini donation_money_007-p002-c0002-q01
Judging [14/50] gpt-4o-mini donation_money_007-p002-c0003-q01
Judging [15/50] gpt-4o-mini donation_money_007-p003-c0004-q01
Judging [16/50] gpt-4o-mini donation_money_008-p001-c0000-q01
Judging [17/50] gpt-4o-mini d

In [13]:
def summarize(records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    summaries = []
    for model in sorted({row["answer_model"] for row in records}):
        model_rows = [row for row in records if row["answer_model"] == model]
        answer_count = len(model_rows)
        relevant_count = sum(1 for row in model_rows if row.get("relevant"))
        total_tokens = sum(int(row.get("total_tokens") or 0) for row in model_rows)
        judge_tokens = sum(int(row.get("judge_total_tokens") or 0) for row in model_rows)
        failures = sum(1 for row in model_rows if row.get("rag_error"))
        summaries.append(
            {
                "answer_model": model,
                "judge_model": JUDGE_MODEL,
                "answer_count": answer_count,
                "relevant_count": relevant_count,
                "irrelevant_count": answer_count - relevant_count,
                "relevance_rate": relevant_count / answer_count if answer_count else 0.0,
                "average_relevance_score": (
                    sum(float(row.get("score") or 0) for row in model_rows) / answer_count
                    if answer_count
                    else 0.0
                ),
                "rag_failures": failures,
                "answer_tokens": total_tokens,
                "judge_tokens": judge_tokens,
            }
        )
    return sorted(summaries, key=lambda row: (row["relevance_rate"], row["average_relevance_score"]), reverse=True)


summary_rows = summarize(judged_records)
detail_columns = [
    "answer_model",
    "question_id",
    "question",
    "answer",
    "relevant",
    "score",
    "rationale",
    "expected_chunk_id",
    "expected_document_id",
    "response_time_seconds",
    "total_tokens",
    "judge_total_tokens",
]

try:
    import pandas as pd

    summary_df = pd.DataFrame(summary_rows)
    details_df = pd.DataFrame(judged_records)
    display(summary_df)
    display(details_df[detail_columns].sort_values(["answer_model", "question_id"]))
except Exception:
    print(json.dumps(summary_rows, indent=2, ensure_ascii=False))
    print(json.dumps([{key: row.get(key) for key in detail_columns} for row in judged_records], indent=2, ensure_ascii=False))


,answer_model,judge_model,answer_count,relevant_count,irrelevant_count,relevance_rate,average_relevance_score,rag_failures,answer_tokens,judge_tokens
0,gpt-4.1,gpt-4o-mini,25,18,7,0.72,2.16,0,41858,14276
1,gpt-4o-mini,gpt-4o-mini,25,15,10,0.60,1.80,0,40836,13267


,answer_model,question_id,question,answer,relevant,score,rationale,expected_chunk_id,expected_document_id,response_time_seconds,total_tokens,judge_total_tokens
25,gpt-4.1,certificate_016-p001-c0000-q01,What is the document ID of the notarial certif...,The document ID of the notarial certificate is...,True,3,The answer directly states the document ID as ...,certificate_016-p001-c0000,certificate_016,8.874,1669,441
26,gpt-4.1,certificate_016-p001-c0001-q01,What is the date of birth of Hendrik Janssen?,Hendrik Janssen was born on 15 March 1948 [cer...,True,3,The answer directly provides the date of birth...,certificate_016-p001-c0001,certificate_016,11.532,2230,550
27,gpt-4.1,certificate_016-p001-c0002-q01,Who are the parents of Thomas Janssen and Emma...,The parents of Thomas Janssen and Emma Janssen...,True,3,The answer directly addresses the question abo...,certificate_016-p001-c0002,certificate_016,9.201,622,467
28,gpt-4.1,certificate_016-p002-c0003-q01,Who is the notary public mentioned in the docu...,The notary public mentioned in the document is...,False,0,The answer incorrectly names the notary public...,certificate_016-p002-c0003,certificate_016,8.854,1369,482
29,gpt-4.1,donation_money_006-p001-c0000-q01,What is the document ID of the notarial deed o...,The document ID of the notarial deed of donati...,True,3,The answer directly provides the document ID o...,donation_money_006-p001-c0000,donation_money_006,33.826,2512,843
30,gpt-4.1,donation_money_006-p001-c0001-q01,Who are referred to as the 'Parties' in the text?,"The term ""Parties"" in the text refers collecti...",True,3,The answer directly explains who the 'Parties'...,donation_money_006-p001-c0001,donation_money_006,8.895,1771,475
31,gpt-4.1,donation_money_006-p002-c0002-q01,What is the amount of the monetary gift donate...,The amount of the monetary gift donated by the...,True,3,The answer directly states the amount of the m...,donation_money_006-p002-c0002,donation_money_006,8.247,1614,536
32,gpt-4.1,donation_money_006-p002-c0003-q01,What laws govern the treatment of the donation...,The donation mentioned is governed by Belgian ...,True,3,The answer directly addresses the question abo...,donation_money_006-p002-c0003,donation_money_006,9.292,1830,778
33,gpt-4.1,donation_money_006-p002-c0004-q01,Who are the donors mentioned in the text?,The donors mentioned in the text are:\n\n- Hen...,True,3,The answer explicitly names the donors as Hend...,donation_money_006-p002-c0004,donation_money_006,7.928,1325,439
34,gpt-4.1,donation_money_006-p003-c0005-q01,Who is the Notary Public mentioned in the docu...,The Notary Public mentioned in the document is...,False,0,The answer provides incorrect information abou...,donation_money_006-p003-c0005,donation_money_006,8.614,1369,452


In [ ]:
# Optional: inspect the weakest answers first.
weak_rows = sorted(judged_records, key=lambda row: (int(row.get("score") or 0), row["answer_model"], row["question_id"]))

for row in weak_rows[:10]:
    print("=" * 100)
    print(f"MODEL: {row['answer_model']} | SCORE: {row.get('score')} | RELEVANT: {row.get('relevant')}")
    print(f"QUESTION: {row['question']}")
    print(f"ANSWER: {row.get('answer', '')}")
    print(f"JUDGE: {row.get('rationale', '')}")
    if row.get("missing_or_wrong_facts"):
        print(f"ISSUES: {row['missing_or_wrong_facts']}")
